In [1]:
%pip install -q mlflow scikit-learn pandas requests

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 50.2/50.2 kB 2.1 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 51.1/51.1 kB 2.4 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.3/44.3 kB 1.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.0/12.0 MB 86.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.1/4.1 MB 99.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.0/2.0 MB 68.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 268.7/268.7 kB 18.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 148.8/148.8 kB 11.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 114.9/114.9 kB 9.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 228.4/228.4 kB 15.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 136.5/136.5 kB 10.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 148.7/148.7 kB 9.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

In [2]:
import os
import time
import subprocess
import requests
import pandas as pd
import mlflow
import mlflow.sklearn

from sklearn.datasets import load_breast_cancer
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, precision_score, recall_score
from mlflow.models import infer_signature

print("MLflow version:", mlflow.__version__)
print("Setup successful!")

MLflow version: 3.17.1
Setup successful!


In [3]:
mlflow.set_tracking_uri("sqlite:///mlflow_w11d2.db")

experiment_name = "W11D2_Model_Registry"
mlflow.set_experiment(experiment_name)

print("Tracking URI:", mlflow.get_tracking_uri())
print("Experiment:", experiment_name)

2026/10/13 12:16:51 INFO mlflow.store.db.utils: Creating initial MLflow database tables...
2026/10/13 12:16:51 INFO mlflow.store.db.utils: Updating database tables
2026/10/13 12:16:55 INFO mlflow.tracking.fluent: Experiment with name 'W11D2_Model_Registry' does not exist. Creating a new experiment.


Tracking URI: sqlite:///mlflow_w11d2.db
Experiment: W11D2_Model_Registry


In [4]:
data = load_breast_cancer(as_frame=True)

X = data.data
y = data.target

X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

print("Training samples:", len(X_train))
print("Testing samples:", len(X_test))
print("Features:", X.shape[1])

Training samples: 455
Testing samples: 114
Features: 30


In [5]:
def run_experiment(n_estimators, max_depth, min_samples_split):
    model = RandomForestClassifier(
        n_estimators=n_estimators,
        max_depth=max_depth,
        min_samples_split=min_samples_split,
        random_state=42,
        n_jobs=-1
    )

    model.fit(X_train, y_train)
    predictions = model.predict(X_test)

    accuracy = accuracy_score(y_test, predictions)
    precision = precision_score(y_test, predictions, zero_division=0)
    recall = recall_score(y_test, predictions, zero_division=0)

    with mlflow.start_run(
        run_name=f"RF_{n_estimators}_{max_depth}_{min_samples_split}"
    ):
        mlflow.log_params({
            "n_estimators": n_estimators,
            "max_depth": max_depth if max_depth is not None else "None",
            "min_samples_split": min_samples_split
        })

        mlflow.log_metrics({
            "accuracy": accuracy,
            "precision": precision,
            "recall": recall
        })

        mlflow.sklearn.log_model(
            sk_model=model,
            name="model",
            signature=infer_signature(X_train, model.predict(X_train)),
            input_example=X_train.head(5),
            skops_trusted_types=["sklearn.tree._tree.Tree"]
        )

        run_id = mlflow.active_run().info.run_id

    return {
        "n_estimators": n_estimators,
        "max_depth": max_depth if max_depth is not None else "None",
        "min_samples_split": min_samples_split,
        "accuracy": accuracy,
        "precision": precision,
        "recall": recall,
        "run_id": run_id,
        "model": model
    }

print("Experiment function defined successfully!")

Experiment function defined successfully!


In [6]:
# Cell 6: Run five experiments

experiment_configs = [
    (50, None, 2),
    (100, None, 2),
    (150, 10, 2),
    (200, 15, 4),
    (100, 8, 5)
]

experiment_results = []

for config in experiment_configs:
    result = run_experiment(*config)
    experiment_results.append(result)
    print(
        f"Completed: n_estimators={result['n_estimators']}, "
        f"max_depth={result['max_depth']}, "
        f"accuracy={result['accuracy']:.4f}"
    )

print("\nTotal experiments completed:", len(experiment_results))

Completed: n_estimators=50, max_depth=None, accuracy=0.9561
Completed: n_estimators=100, max_depth=None, accuracy=0.9561
Completed: n_estimators=150, max_depth=10, accuracy=0.9561
Completed: n_estimators=200, max_depth=15, accuracy=0.9474
Completed: n_estimators=100, max_depth=8, accuracy=0.9474

Total experiments completed: 5


In [7]:
# Cell 7: Compare results

results_df = pd.DataFrame([
    {k: v for k, v in result.items() if k != "model"}
    for result in experiment_results
])

results_df = results_df.sort_values(
    "accuracy", ascending=False
).reset_index(drop=True)

display(results_df)

best_result = max(experiment_results, key=lambda x: x["accuracy"])
best_model = best_result["model"]

print("Best model parameters:")
print({
    "n_estimators": best_result["n_estimators"],
    "max_depth": best_result["max_depth"],
    "min_samples_split": best_result["min_samples_split"]
})
print("Best accuracy:", round(best_result["accuracy"], 4))

,n_estimators,max_depth,min_samples_split,accuracy,precision,recall,run_id
0,50,None,2,0.956140,0.958904,0.972222,8d14bd54d3fc4117b0492c342e53e8b6
1,100,None,2,0.956140,0.958904,0.972222,f24aa99901a540418ec9db0fc0cc45bc
2,150,10,2,0.956140,0.958904,0.972222,14585cc4c79748009837d5aaec6ec24c
3,200,15,4,0.947368,0.958333,0.958333,2f6ab46394e14a79a4f30b818539d346
4,100,8,5,0.947368,0.958333,0.958333,d181311f9ec945b2905ceec5da54f568


Best model parameters:
{'n_estimators': 50, 'max_depth': 'None', 'min_samples_split': 2}
Best accuracy: 0.9561


In [8]:
# Cell 8: Register the best model

model_name = "W11D2_Best_RandomForest"

with mlflow.start_run(run_name="best_model_registration"):
    model_info = mlflow.sklearn.log_model(
        sk_model=best_model,
        name="model",
        signature=infer_signature(X_train, best_model.predict(X_train)),
        input_example=X_train.head(5),
        skops_trusted_types=["sklearn.tree._tree.Tree"]
    )

registered_model = mlflow.register_model(
    model_uri=model_info.model_uri,
    name=model_name
)

model_uri = f"models:/{model_name}/{registered_model.version}"

print("Model registered successfully!")
print("Model name:", registered_model.name)
print("Version:", registered_model.version)
print("Model URI:", model_uri)

Model registered successfully!
Model name: W11D2_Best_RandomForest
Version: 1
Model URI: models:/W11D2_Best_RandomForest/1


Successfully registered model 'W11D2_Best_RandomForest'.
Created version '1' of model 'W11D2_Best_RandomForest'.


In [9]:
# Cell 9: Load registered version and test predictions

loaded_model = mlflow.sklearn.load_model(model_uri)

predictions = loaded_model.predict(X_test)

print("Registered model loaded successfully!")
print("Test samples:", len(X_test))
print("First 10 predictions:", predictions[:10])
print("Test accuracy:", round(accuracy_score(y_test, predictions), 4))

assert len(predictions) == len(y_test)
print("Prediction validation passed!")

Registered model loaded successfully!
Test samples: 114
First 10 predictions: [0 1 0 0 0 1 1 0 0 0]
Test accuracy: 0.9561
Prediction validation passed!


In [10]:
# Cell 10: Export experiment comparison

results_df.to_csv("w11d2_experiment_results.csv", index=False)

print("Experiment results saved!")
print("Total experiments:", len(results_df))
display(results_df)

assert len(results_df) == 5
print("Validation passed!")

Experiment results saved!
Total experiments: 5


,n_estimators,max_depth,min_samples_split,accuracy,precision,recall,run_id
0,50,None,2,0.956140,0.958904,0.972222,8d14bd54d3fc4117b0492c342e53e8b6
1,100,None,2,0.956140,0.958904,0.972222,f24aa99901a540418ec9db0fc0cc45bc
2,150,10,2,0.956140,0.958904,0.972222,14585cc4c79748009837d5aaec6ec24c
3,200,15,4,0.947368,0.958333,0.958333,2f6ab46394e14a79a4f30b818539d346
4,100,8,5,0.947368,0.958333,0.958333,d181311f9ec945b2905ceec5da54f568


Validation passed!


In [11]:
# Cell 11: Start model serving

import subprocess
import time

serve_log = open("mlflow_w11d2_serve.log", "w")

serve_process = subprocess.Popen(
    [
        "mlflow", "models", "serve",
        "-m", model_uri,
        "-p", "5002",
        "--host", "127.0.0.1",
        "--env-manager", "local"
    ],
    stdout=serve_log,
    stderr=subprocess.STDOUT
)

time.sleep(15)

if serve_process.poll() is None:
    print("MLflow server process is running.")
    print("Endpoint: http://127.0.0.1:5002/invocations")
else:
    serve_log.flush()
    print("Server failed to start. Log output:")
    print(open("mlflow_w11d2_serve.log").read()[-4000:])

MLflow server process is running.
Endpoint: http://127.0.0.1:5002/invocations


In [12]:
# Cell 12: Send prediction request

import requests
import time

payload = {
    "dataframe_split": X_test.head(5).to_dict(orient="split")
}

response = None

for attempt in range(15):
    try:
        response = requests.post(
            "http://127.0.0.1:5002/invocations",
            json=payload,
            timeout=30
        )
        break
    except requests.exceptions.RequestException:
        print(f"Waiting for server... {attempt + 1}/15")
        time.sleep(4)

if response is None:
    print("Could not connect to the server.")
    serve_log.flush()
    print(open("mlflow_w11d2_serve.log").read()[-4000:])
else:
    print("HTTP status:", response.status_code)
    print(response.text[:2000])
    response.raise_for_status()
    print("API test passed!")

HTTP status: 200
{"predictions": [0, 1, 0, 0, 0]}
API test passed!


In [13]:
# Cell 13: Create submission files

readme = """# W11D2: MLflow Model Registry & Versioning

## Objective
Track five scikit-learn experiments, compare their metrics,
register the best model, load its registered version, and serve it.

## Implemented
- MLflow experiment tracking
- Hyperparameter logging and metric comparison
- Model artifact logging
- Best model registration and versioning
- Registered model loading and prediction testing
- MLflow model-serving API test

## Files
- w11d2_experiment_results.csv
- README_W11D2.md
- requirements_W11D2.txt
"""

with open("README_W11D2.md", "w") as f:
    f.write(readme)

with open("requirements_W11D2.txt", "w") as f:
    f.write("mlflow\\nscikit-learn\\npandas\\nrequests\\n")

print("Created README_W11D2.md")
print("Created requirements_W11D2.txt")

Created README_W11D2.md
Created requirements_W11D2.txt


In [14]:
# Cell 14: Validate deliverables

import os

required_files = [
    "w11d2_experiment_results.csv",
    "README_W11D2.md",
    "requirements_W11D2.txt"
]

for filename in required_files:
    print(filename, "EXISTS" if os.path.exists(filename) else "MISSING")

print("\nExperiments:", len(experiment_results))
print("Registered model:", registered_model.name)
print("Registered version:", registered_model.version)

assert len(experiment_results) == 5
assert all(os.path.exists(f) for f in required_files)

print("\nW11D2 validation completed!")

w11d2_experiment_results.csv EXISTS
README_W11D2.md EXISTS
requirements_W11D2.txt EXISTS

Experiments: 5
Registered model: W11D2_Best_RandomForest
Registered version: 1

W11D2 validation completed!


In [15]:

from google.colab import files

files.download("w11d2_experiment_results.csv")

files.download("requirements_W11D2.txt")


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>